In [ ]:
%%configure -f
{"vCores": 16, "defaultLakehouse": {"name": "<YOUR_LAKEHOUSE_NAME>", "id": "<YOUR_LAKEHOUSE_ID>", "workspaceId": "<YOUR_WORKSPACE_ID>"}}


# fabric-rlm document-analysis direct-vs-RLM evaluation

Fabric-native adaptation of `Trampoline-AI/predict-rlm/examples/document_analysis`.
Downloads the YYJ-2025 Parking Management RFP (5.4 MB, 136 pages) — the very PDF
predict-rlm ships in `sample/input/` — runs a **direct single-call baseline**
against the extracted text, then runs **`fabric_rlm.RLM`** with the existing
`pdf_document_analysis` skill, and writes comparable JSON/Markdown artifacts.

This is the example the `pdf_document_analysis` skill was originally written for
(single long doc, briefing-style report) — the most direct test of skill quality.

| Mode | Trigger | LM | Output root |
|---|---|---|---|
| **Fabric** | `/lakehouse/default` exists | `FabricChatLM` (notebook identity) | `/lakehouse/default/Files/fabric_rlm_document_analysis/<run_id>/` |
| **Local** | otherwise | `fabric_rlm.OpenAILM` (`OPENAI_API_KEY`) | `./_local_runs/document_analysis/<run_id>/` |

Coverage scored against ~30 anchors mined from
`examples/document_analysis/sample/output/report.md` in the predict-rlm repo,
grouped by entities, dates, money, key concepts, and contract terms.


In [ ]:
DA_RUN_ID = ''
DA_MODEL = 'gpt-5'
DA_MAX_TURNS = 25
DA_DIRECT_TEXT_CHARS = 200000   # ~200KB cap for the direct baseline excerpt
DA_PDF_URLS = [
    'https://raw.githubusercontent.com/Trampoline-AI/predict-rlm/2d93675d6d69b45f9eda9b8fc01e178323f8e6cb/examples/document_analysis/sample/input/YYJ-2025-Parking-Management-RFP.pdf',
]
DA_TIMEOUT_SECONDS = 1500


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, time, hashlib, shutil, traceback, uuid

LAKEHOUSE_ROOT = Path('/lakehouse/default')
FABRIC_RUNTIME = LAKEHOUSE_ROOT.exists() and (LAKEHOUSE_ROOT / 'Files').exists()

RUN_ID = str(globals().get('DA_RUN_ID') or '').strip() or (
    time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:8]
)

if FABRIC_RUNTIME:
    FILES_ROOT = LAKEHOUSE_ROOT / 'Files'
    RUN_ROOT = FILES_ROOT / 'fabric_rlm_document_analysis' / RUN_ID
else:
    FILES_ROOT = Path.cwd() / '_local_runs'
    RUN_ROOT = FILES_ROOT / 'document_analysis' / RUN_ID

INPUT_ROOT = RUN_ROOT / 'input'
DIRECT_ROOT = RUN_ROOT / 'direct'
RLM_ROOT = RUN_ROOT / 'rlm'
TRAJECTORY_ROOT = RUN_ROOT / 'trajectories'
for folder in [RUN_ROOT, INPUT_ROOT, DIRECT_ROOT, RLM_ROOT, TRAJECTORY_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

STAGE_EVENTS_PATH = RUN_ROOT / 'stage_events.jsonl'

def write_stage(stage, **details):
    payload = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'stage': stage, **details}
    with open(STAGE_EVENTS_PATH, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')
        handle.flush()
    print(f'STAGE {stage}: {details}')

def write_json(relative_path, payload):
    path = RUN_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    return path

write_stage('bootstrap_start', run_root=str(RUN_ROOT), runtime='fabric' if FABRIC_RUNTIME else 'local')
print(f'Document-analysis run root: {RUN_ROOT}')

EXPECTED_SKILLS = ['pdf_document_analysis', 'validation', 'error_handling']

def _imports_ok():
    try:
        import fabric_rlm  # noqa: F401
        from fabric_rlm import File, RLM  # noqa: F401
        import fitz  # noqa: F401
        import nest_asyncio  # noqa: F401
        return True
    except Exception as exc:
        write_stage('import_probe_failed', error=repr(exc))
        return False

if FABRIC_RUNTIME:
    LOCAL_WHEEL_PATH = Path(os.environ.get('FABRIC_RLM_WHEEL', str(FILES_ROOT / 'fabric_rlm_longcot' / 'wheels' / 'fabric_rlm-0.2.5-py3-none-any.whl')))
    ISOLATED_DEPS_TARGET = Path(os.environ.get('FABRIC_RLM_DA_DEPS_TARGET', str(FILES_ROOT / 'fabric_rlm_document_analysis' / '_deps' / (LOCAL_WHEEL_PATH.stem + '-da'))))

    def _prepend_path(p):
        s = str(p)
        if s not in sys.path: sys.path.insert(0, s)
        existing = os.environ.get('PYTHONPATH', '')
        if not existing or existing.split(os.pathsep)[0] != s:
            os.environ['PYTHONPATH'] = s + (os.pathsep + existing if existing else '')

    def _clear_modules(prefixes):
        for name in list(sys.modules):
            if any(name == p or name.startswith(p + '.') for p in prefixes):
                del sys.modules[name]

    if not ISOLATED_DEPS_TARGET.exists() and LOCAL_WHEEL_PATH.exists():
        ISOLATED_DEPS_TARGET.mkdir(parents=True, exist_ok=True)
        write_stage('bootstrap_install_start', wheel=str(LOCAL_WHEEL_PATH), target=str(ISOLATED_DEPS_TARGET))
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall',
                               '--no-deps', '--target', str(ISOLATED_DEPS_TARGET), str(LOCAL_WHEEL_PATH)])
    if ISOLATED_DEPS_TARGET.exists():
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fabric_rlm'])

    if not _imports_ok():
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
                               '--target', str(ISOLATED_DEPS_TARGET), 'pymupdf>=1.24.0,<1.25', 'nest_asyncio>=1.6'])
        _prepend_path(ISOLATED_DEPS_TARGET)
        _clear_modules(['fitz', 'pymupdf'])

    if not _imports_ok():
        raise ImportError('fabric_rlm/pymupdf unavailable after Fabric bootstrap')
else:
    if not _imports_ok():
        print('Local imports missing — attempting `pip install -e . pymupdf` ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(Path.cwd()), 'pymupdf>=1.24.0', 'nest_asyncio>=1.6'])
        for k in list(sys.modules):
            if k == 'fabric_rlm' or k.startswith('fabric_rlm.') or k in {'fitz', 'pymupdf'}:
                del sys.modules[k]
        if not _imports_ok():
            raise ImportError('fabric_rlm/pymupdf unavailable after local bootstrap')

import fabric_rlm
available = set(fabric_rlm.list_skills())
missing = set(EXPECTED_SKILLS) - available
if missing:
    raise RuntimeError(f'fabric_rlm at {fabric_rlm.__file__} missing skills: {sorted(missing)}; available={sorted(available)}')
write_stage('bootstrap_ok', fabric_rlm=str(fabric_rlm.__file__), version=getattr(fabric_rlm, '__version__', '?'),
            available_skills=sorted(available), runtime='fabric' if FABRIC_RUNTIME else 'local')


In [ ]:
import re, urllib.error, urllib.request
import fitz
import fabric_rlm
from fabric_rlm import File, RLM

MODEL = str(globals().get('DA_MODEL') or 'gpt-5')
MAX_TURNS = int(globals().get('DA_MAX_TURNS') or 25)
DIRECT_TEXT_CHARS = int(globals().get('DA_DIRECT_TEXT_CHARS') or 200000)
PDF_URLS = list(globals().get('DA_PDF_URLS') or [])
TIMEOUT_SECONDS = int(globals().get('DA_TIMEOUT_SECONDS') or 1500)
MODEL_KWARGS = {'max_tokens': 16000, 'temperature': 1.0}
_model_short = MODEL.split('/')[-1].lower()
_is_reasoning = _model_short.startswith(('gpt-5', 'o1', 'o3', 'o4')) and not _model_short.startswith('gpt-5-chat')
if _is_reasoning:
    MODEL_KWARGS = {'max_tokens': 32000, 'reasoning_effort': 'low'}

if len(PDF_URLS) < 1:
    raise ValueError('DA_PDF_URLS must list at least one PDF URL.')

OUTPUT_SCHEMA_HINT = '''
Return an object with these keys:
  - report (string, full markdown briefing report — Executive Summary,
    Key Dates and Timeline, Key Entities and Stakeholders, Financial
    Information, Contract Terms / Other relevant sections)
  - key_dates (list of {name, date, time?, timezone?}; date is ISO YYYY-MM-DD;
    time is 24-hour HH:MM if applicable)
  - key_entities (list of {name, role?, contact?})
  - summary (string, 1-paragraph executive summary)
'''.strip()

CRITERIA = '''
Analyze the provided PDF document(s) (a Request for Proposals plus its
schedules and appendices) and produce a structured briefing report.
1. Survey the document(s) — file names, page counts, document types.
2. Render relevant pages as images and use predict() to extract content
   (cover page, table of contents, key clauses, schedules, appendices,
   contract execution block, fee schedules).
3. Produce a markdown report with these sections (use tables for structured
   data, prose for analysis):
   - Executive Summary
   - Key Dates and Timeline (table: Date | Time | Event | Notes)
   - Key Entities and Stakeholders (table: Entity | Role | Contact)
   - Financial Information (rate tables, security/insurance, fees,
     payment terms, contract term highlights)
   - Any other sections the document warrants
4. Populate key_dates and key_entities as structured lists in addition to
   the report. Dates must be ISO YYYY-MM-DD.
'''.strip()

EXPECTED_ANCHORS = {
    # Entities and people named in the reference report.
    'entities': [
        'Victoria Airport Authority', 'Victoria International Airport', 'YYJ',
        'David Parson', 'Elizabeth M. Brown',
    ],
    # Dates that the reference report calls out specifically.
    'dates': [
        '2025-02-13', '2025-02-20', '2025-02-27', '2025-03-06',
        '2025-03-31', '2025-04-24', '2025-06-01',
        '2030-05-31', '2029-12-01', '2030-06-01',
    ],
    # Money / numeric anchors from rate and security tables.
    'money': [
        '4.00', '1.00', '18.00', '9.00', '200.00', '540.00',
        '5%', '10,000', '26.824',
    ],
    # Key concepts/terms the briefing must surface.
    'concepts': [
        'irrevocable letter of credit', 'mandatory pre-bid meeting',
        'gross revenue', 'workSafeBC', 'management fee',
        'initial term', 'renewal term', 'parking',
        '60 days', 'sixty (60) days',
    ],
}

if FABRIC_RUNTIME:
    class FabricChatLM:
        def __init__(self, model, timeout=360, **default_kwargs):
            from synapse.ml.fabric.service_discovery import get_fabric_env_config
            from synapse.ml.fabric.token_utils import TokenUtils
            env = get_fabric_env_config().fabric_env_config
            base = f'{env.ml_workload_endpoint}cognitive/openai'.rstrip('/')
            self.model = model
            self.timeout = timeout
            self.default_kwargs = dict(default_kwargs)
            self.headers = {'Authorization': TokenUtils().get_openai_auth_header(), 'Content-Type': 'application/json'}
            self.urls = [
                f'{base}/openai/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
                f'{base}/deployments/{model}/chat/completions?api-version=2025-04-01-preview',
            ]
        def __call__(self, *, messages, **kwargs):
            ck = {**self.default_kwargs, **kwargs}
            body = {'messages': messages}
            if ck.get('temperature') is not None: body['temperature'] = ck['temperature']
            if ck.get('max_tokens') is not None: body['max_completion_tokens'] = int(ck['max_tokens'])
            data = json.dumps(body).encode('utf-8')
            errors = []
            for i, url in enumerate(self.urls):
                req = urllib.request.Request(url, data=data, headers=self.headers, method='POST')
                try:
                    with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                        payload = json.loads(resp.read().decode('utf-8'))
                    choice = (payload.get('choices') or [{}])[0]
                    msg = choice.get('message') or {}
                    return {'content': msg.get('content') or choice.get('text') or '',
                            'usage': payload.get('usage') or {}, 'model': payload.get('model') or self.model}
                except urllib.error.HTTPError as exc:
                    detail = exc.read().decode('utf-8', errors='replace')[:2000]
                    errors.append({'url_index': i, 'status': exc.code, 'detail': detail})
                    if exc.code not in {400, 404}: break
                except Exception as exc:
                    errors.append({'url_index': i, 'error': repr(exc)}); break
            raise RuntimeError(f'FabricChatLM call failed: {errors}')

    def make_direct_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return FabricChatLM(MODEL, timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    SUB_LM_SPEC = 'fabric/' + MODEL
else:
    def _detect_local_provider():
        if os.environ.get('OPENAI_API_KEY'): return 'openai'
        if os.environ.get('OPENROUTER_API_KEY'): return 'openrouter'
        raise RuntimeError('Local runtime requires OPENAI_API_KEY or OPENROUTER_API_KEY.')
    def _local_model_spec():
        provider = _detect_local_provider()
        if provider == 'openai': return MODEL
        if MODEL.startswith('openrouter/'): return MODEL
        if '/' in MODEL: return 'openrouter/' + MODEL
        return 'openrouter/openai/' + MODEL
    def _make_local_lm(**extra_kwargs):
        import dspy
        provider = _detect_local_provider()
        kwargs = dict(MODEL_KWARGS); kwargs.update(extra_kwargs)
        if provider == 'openai':
            from fabric_rlm import OpenAILM
            return OpenAILM(MODEL, **kwargs)
        return dspy.LM(model=_local_model_spec(),
                       api_key=os.environ['OPENROUTER_API_KEY'],
                       api_base='https://openrouter.ai/api/v1', **kwargs)
    class _LocalDirectLM:
        def __init__(self, timeout=360, **default_kwargs):
            self.model = _local_model_spec(); self.timeout = timeout
            self._lm = _make_local_lm(**default_kwargs)
        def __call__(self, *, messages, **_kwargs):
            out = self._lm(messages=messages)
            if isinstance(out, list) and out:
                first = out[0]
                if isinstance(first, str): return {'content': first, 'usage': {}, 'model': self.model}
                if isinstance(first, dict): return {'content': first.get('content', ''), 'usage': {}, 'model': self.model}
            if isinstance(out, str): return {'content': out, 'usage': {}, 'model': self.model}
            return {'content': str(out), 'usage': {}, 'model': self.model}
    def make_direct_lm():
        _detect_local_provider(); return _LocalDirectLM(timeout=TIMEOUT_SECONDS, **MODEL_KWARGS)
    def make_rlm_lm():
        return _make_local_lm()
    try: SUB_LM_SPEC = _local_model_spec()
    except Exception: SUB_LM_SPEC = ('openai/' + MODEL) if '/' not in MODEL else MODEL

def response_to_text(response):
    if isinstance(response, str): return response
    if isinstance(response, dict): return str(response.get('content') or response.get('text') or response)
    return str(getattr(response, 'content', response))

def extract_json_object(text):
    cleaned = text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find('{'); end = cleaned.rfind('}')
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end + 1])
        raise

def normalize_items(value):
    if value is None: return []
    if isinstance(value, list): return value
    return [value]

def score_analysis(analysis):
    report = str(analysis.get('report') or analysis.get('markdown_report') or '')
    key_dates = normalize_items(analysis.get('key_dates'))
    key_entities = normalize_items(analysis.get('key_entities'))
    summary = str(analysis.get('summary') or '')
    corpus = '\n'.join([
        report, summary,
        json.dumps(key_dates, ensure_ascii=False, default=str),
        json.dumps(key_entities, ensure_ascii=False, default=str),
    ]).lower()

    groups = {}
    total_hits = 0
    total_possible = 0
    for group, anchors in EXPECTED_ANCHORS.items():
        hits = [a for a in anchors if a.lower() in corpus]
        misses = [a for a in anchors if a.lower() not in corpus]
        groups[group] = {'hits': hits, 'misses': misses, 'score': len(hits), 'possible': len(anchors)}
        total_hits += len(hits); total_possible += len(anchors)

    return {
        'score': total_hits, 'possible': total_possible,
        'coverage': total_hits / max(total_possible, 1),
        'groups': groups,
        'report_chars': len(report),
        'key_date_count': len(key_dates),
        'key_entity_count': len(key_entities),
        'summary_chars': len(summary),
    }


In [ ]:
pdf_paths = []
for url in PDF_URLS:
    name = url.rsplit('/', 1)[-1]
    target = INPUT_ROOT / name
    if not target.exists():
        write_stage('download_pdf_start', url=url, path=str(target))
        req = urllib.request.Request(url, headers={'User-Agent': 'fabric-rlm-document-analysis'})
        with urllib.request.urlopen(req, timeout=300) as resp:
            target.write_bytes(resp.read())
        write_stage('download_pdf_done', bytes=target.stat().st_size, path=str(target))
    else:
        write_stage('download_pdf_cached', path=str(target), bytes=target.stat().st_size)
    pdf_paths.append(target)

pdf_text_blocks = []
pdf_metadata = []
for p in pdf_paths:
    doc = fitz.open(p)
    pages = [{'page': i + 1, 'text': page.get_text('text')} for i, page in enumerate(doc)]
    doc.close()
    full_text = '\n\n'.join(f'--- Page {row["page"]} ---\n{row["text"]}' for row in pages)
    text_path = INPUT_ROOT / (p.stem + '.txt')
    text_path.write_text(full_text, encoding='utf-8')
    excerpt = full_text[:DIRECT_TEXT_CHARS]
    pdf_text_blocks.append((p.name, excerpt))
    pdf_metadata.append({
        'name': p.name, 'pdf_path': str(p), 'text_path': str(text_path),
        'page_count': len(pages), 'text_chars': len(full_text),
        'direct_excerpt_chars': len(excerpt),
    })

manifest = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'source': 'Trampoline-AI/predict-rlm examples/document_analysis sample PDF',
    'pdf_urls': PDF_URLS,
    'documents': pdf_metadata,
    'model': MODEL, 'max_turns': MAX_TURNS,
    'fabric_rlm_version': getattr(fabric_rlm, '__version__', 'unknown'),
    'python': sys.version, 'platform': platform.platform(),
}
write_json('manifest.json', manifest)
write_stage('documents_ready',
            docs=[{'name': m['name'], 'pages': m['page_count'], 'text_chars': m['text_chars']}
                  for m in pdf_metadata])


In [ ]:
direct_lm = make_direct_lm()
direct_user_parts = [CRITERIA, OUTPUT_SCHEMA_HINT,
                     f'You are receiving the extracted text of one or more PDF documents to analyze. '
                     f'This is a single-pass direct baseline; use only this text. The excerpt is '
                     f'truncated to the first {DIRECT_TEXT_CHARS:,} characters per document.']
for name, excerpt in pdf_text_blocks:
    direct_user_parts.append(f'\n=== DOCUMENT: {name} ===\n{excerpt}')
direct_messages = [
    {'role': 'system',
     'content': ('You analyze documents and return ONLY valid JSON with keys '
                 'report, key_dates, key_entities, summary. report is a markdown string '
                 'with sections; key_dates and key_entities are arrays of objects per the schema.')},
    {'role': 'user', 'content': '\n\n'.join(direct_user_parts)},
]
write_stage('direct_start', model=MODEL,
            total_text_chars=sum(len(e) for _, e in pdf_text_blocks))
direct_started = time.perf_counter()
direct_response = direct_lm(messages=direct_messages)
direct_duration = time.perf_counter() - direct_started
direct_text = response_to_text(direct_response)
(DIRECT_ROOT / 'raw_response.txt').write_text(direct_text, encoding='utf-8')
try:
    direct_analysis = extract_json_object(direct_text)
    direct_error = None
except Exception as exc:
    direct_analysis = {'report': direct_text, 'key_dates': [], 'key_entities': [], 'summary': ''}
    direct_error = repr(exc)
direct_score = score_analysis(direct_analysis)
if direct_analysis.get('report'):
    (DIRECT_ROOT / 'report.md').write_text(str(direct_analysis['report']), encoding='utf-8')
write_json('direct/analysis.json', direct_analysis)
write_json('direct/score.json', direct_score)
write_stage('direct_done', duration_s=direct_duration,
            score=direct_score['score'], possible=direct_score['possible'],
            coverage=round(direct_score['coverage'], 3),
            key_date_count=direct_score['key_date_count'],
            key_entity_count=direct_score['key_entity_count'],
            parse_error=direct_error)
print(f"DIRECT  : score={direct_score['score']}/{direct_score['possible']}  "
      f"coverage={direct_score['coverage']:.0%}  "
      f"key_dates={direct_score['key_date_count']}  "
      f"key_entities={direct_score['key_entity_count']}  "
      f"in {direct_duration:.1f}s")


In [ ]:
RLM_TASK = '''
Produce a structured briefing analysis of the provided PDF document(s).

You have access to the preloaded `pdf_document_analysis` skill — follow it for
long-document work:
- Use Python and PyMuPDF (`fitz`) to open the PDF and record `page_count`.
- Use raw text for keyword search and candidate discovery; render layout-sensitive
  pages (cover page, fee tables, signature blocks, schedules, appendices) at
  ~200 DPI to data URIs and pass them to `predict()`.
- Use `asyncio.gather()` over independent `await predict(...)` calls per
  page or section when it can improve coverage.
- Ground every important fact to a page number.
- All dates in key_dates must be ISO YYYY-MM-DD.

Write a Markdown report to {output_dir}/report.md and a JSON analysis to
{output_dir}/analysis.json.

Before SUBMIT, run a self-check and repair any failure:
- The report has Executive Summary, Key Dates and Timeline, Key Entities and
  Stakeholders, Financial Information sections (use markdown tables for
  structured data).
- key_dates is a list of {{name, date, time?, timezone?}} dicts; dates ISO.
- key_entities is a list of {{name, role?, contact?}} dicts.
- Every claim is backed by content found in the document.
- Numbers (rates, dollar amounts, percentages) are reported verbatim from
  the source.

Call SUBMIT(report=..., key_dates=..., key_entities=..., summary=...,
            report_path=..., analysis_path=..., page_counts=...).
Required JSON-friendly shapes:
- key_dates: list of dicts with keys name, date, and optional time, timezone.
- key_entities: list of dicts with keys name, and optional role, contact.
- page_counts: dict mapping document filename to page_count.
Do not include unsupported custom objects in SUBMIT.
'''.strip()

write_stage('rlm_setup_start', model=MODEL, max_turns=MAX_TURNS)
try:
    rlm_lm = make_rlm_lm()
    document_files = [File(str(p)) for p in pdf_paths]
    rlm = RLM.task(
        task=RLM_TASK,
        inputs={'documents': document_files, 'criteria': CRITERIA, 'output_dir': str(RLM_ROOT)},
        outputs=['report', 'key_dates', 'key_entities', 'summary',
                 'report_path', 'analysis_path', 'page_counts'],
        lm=rlm_lm,
        sub_lm=SUB_LM_SPEC,
        max_turns=MAX_TURNS,
        skills=['pdf_document_analysis'],
        enable_skill_autoloading=True,
        timeout=TIMEOUT_SECONDS,
    )
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_setup_failed', error=repr(exc), traceback=tb[-4000:])
    (RLM_ROOT / 'setup_error.txt').write_text(tb, encoding='utf-8')
    raise
write_stage('rlm_start', model=MODEL, max_turns=MAX_TURNS, n_documents=len(document_files))
rlm_started = time.perf_counter()
try:
    rlm_result = rlm.run()
except Exception as exc:
    tb = traceback.format_exc()
    write_stage('rlm_run_failed', error=repr(exc), traceback=tb[-4000:],
                duration_s=time.perf_counter() - rlm_started)
    (RLM_ROOT / 'run_error.txt').write_text(tb, encoding='utf-8')
    raise
rlm_duration = time.perf_counter() - rlm_started

trajectory_path = TRAJECTORY_ROOT / 'document_analysis_rlm.jsonl'
rlm_result.trajectory.write_jsonl(trajectory_path)

if rlm_result.submitted and rlm_result.payload:
    rlm_analysis = dict(rlm_result.payload)
else:
    rlm_analysis = {'report': '', 'key_dates': [], 'key_entities': [], 'summary': '',
                    'failure_reason': rlm_result.failure_reason}

rlm_score = score_analysis(rlm_analysis)
if rlm_analysis.get('report'):
    (RLM_ROOT / 'report.md').write_text(str(rlm_analysis['report']), encoding='utf-8')
write_json('rlm/analysis.json', rlm_analysis)
write_json('rlm/score.json', rlm_score)
write_stage('rlm_done', duration_s=rlm_duration, submitted=rlm_result.submitted,
            turns=len(rlm_result.trajectory.turns),
            score=rlm_score['score'], possible=rlm_score['possible'],
            coverage=round(rlm_score['coverage'], 3),
            key_date_count=rlm_score['key_date_count'],
            key_entity_count=rlm_score['key_entity_count'])
print(f"RLM     : submitted={rlm_result.submitted}  turns={len(rlm_result.trajectory.turns)}  "
      f"score={rlm_score['score']}/{rlm_score['possible']}  "
      f"coverage={rlm_score['coverage']:.0%}  "
      f"key_dates={rlm_score['key_date_count']}  "
      f"key_entities={rlm_score['key_entity_count']}  "
      f"in {rlm_duration:.1f}s")


In [ ]:
record = {
    'run_id': RUN_ID, 'runtime': 'fabric' if FABRIC_RUNTIME else 'local',
    'model': MODEL,
    'source_example': 'https://github.com/Trampoline-AI/predict-rlm/tree/main/examples/document_analysis',
    'documents': [m['name'] for m in pdf_metadata],
    'direct': {'duration_s': direct_duration, 'score': direct_score, 'parse_error': direct_error},
    'rlm':    {'duration_s': rlm_duration,    'submitted': rlm_result.submitted,
               'failure_reason': rlm_result.failure_reason, 'score': rlm_score,
               'turns': len(rlm_result.trajectory.turns),
               'trajectory_path': str(trajectory_path)},
}
record['rlm_upgrade'] = (
    rlm_score['score'] > direct_score['score']
    or rlm_score['coverage'] > direct_score['coverage']
    or rlm_score['key_entity_count'] > direct_score['key_entity_count']
)
write_json('record.json', record)

print()
print('=' * 70)
print('DOCUMENT ANALYSIS — DIRECT vs RLM')
print('=' * 70)
for label, sc, dur in [('DIRECT', direct_score, direct_duration),
                       ('RLM   ', rlm_score, rlm_duration)]:
    print(f'{label}:  coverage={sc["coverage"]:.0%}  '
          f'({sc["score"]}/{sc["possible"]} anchors)  '
          f'key_dates={sc["key_date_count"]}  '
          f'key_entities={sc["key_entity_count"]}  '
          f'report_chars={sc["report_chars"]}  '
          f'in {dur:.1f}s')
print()
print('Per-anchor-group hit rate:')
for group in EXPECTED_ANCHORS:
    d = direct_score['groups'][group]; r = rlm_score['groups'][group]
    print(f'  {group:12s}  direct {d["score"]}/{d["possible"]}   rlm {r["score"]}/{r["possible"]}')
    if d['misses'] or r['misses']:
        for label, sc in [('   direct misses', d['misses']), ('   rlm    misses', r['misses'])]:
            if sc: print(f'  {label}: {sc}')
print()
print(f'Run root: {RUN_ROOT}')
print(f'Trajectory: {trajectory_path}')
print(f'RLM upgraded over direct? {record["rlm_upgrade"]}')
